In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from articulate import articulate
from omegaconf import OmegaConf
from dotenv import load_dotenv
import os
from articulate_anything.utils.utils import (
    load_config,
    join_path,
)
from PIL import Image
from articulate_anything.utils.viz import display_code, show_video

In [ ]:
import os
os.chdir("..")

In [ ]:
API_KEY = "YOUR-ACTUAL-API-KEY"
## we have our API key stored in a .env file
load_dotenv()
API_KEY = os.environ.get('API_KEY')

In [ ]:
modality = "text"
prompt = "suitcase with a retractable handle"
task = "suitcase"
overwrite = False
out_dir = join_path("results", modality, task)

In [ ]:
cfg = load_config()
cfg.prompt = prompt
cfg.modality = modality
cfg.out_dir = out_dir
cfg.api_key = API_KEY

cfg.model_name = "claude-3-5-sonnet-20241022"

cfg.joint_actor.mode = "text"
cfg.joint_actor.targetted_affordance = False
cfg.gen_config.overwrite = overwrite

First, make sure that you preprocess the partnet dataset first. Run

```
python articulate_anything/preprocess/preprocess_partnet.py parallel={int} modality=text
```

This will combine meshes belonging to one link to one mesh and also render the links and objects from multiple view for computing CLIP.

In [ ]:
steps = articulate(cfg)

## Mesh Retrieval

Let's inspect the steps starting with the mesh retrieval

In [ ]:
mesh_retrieval = steps["Mesh Retrieval"]

First, the VLM is asked to expand a text prompt into more details (i.e., densify the text)

In [ ]:
mesh_retrieval["Task Specification"].load_prediction()

In [ ]:
mesh_retrieval["Task Specification"].load_prediction()

In [ ]:
mesh_retrieval["Box Layout"].load_prediction()

In [ ]:
mesh_retrieval["Mesh Retrieval"].load_prediction()

In [ ]:
mesh_retrieval["Mesh Retrieval"].cfg.out_dir

## Link Placement

In [ ]:
link_art = steps["Link Articulation"]
link_actor = link_art["Link actor"][0]

Here's the code to place the links in the 3D space

In [ ]:
code_string = link_actor.load_prediction()
display_code(code_string)

Here's a render of the predicted code

In [ ]:
Image.open(join_path(link_actor.cfg.out_dir, "robot_frontview.png"))

## Joint Prediction

In [ ]:
joint_art = steps["Joint Articulation"]
joint_actor = joint_art["Joint actor"][0]

In [ ]:
## find all .mp4 files in the joint_actor output directory
videos = [f for f in os.listdir(joint_actor.cfg.out_dir) if f.endswith(".mp4")]
print(videos)

In [ ]:
video = "video_suitcase_retractable_handle_base_to_suitcase_retractable_handle_frontview.mp4"
show_video(join_path(joint_actor.cfg.out_dir, video),
           use_gif=True)

In [ ]:
code_string = joint_actor.load_prediction()
display_code(code_string)